# AgentLoop
从做一步到做一件事

## 2.1 ReAct Loop

### 2.1 While Loop

In [1]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI


ModuleNotFoundError: No module named 'langchain_openai'

In [ ]:
DEFAULT_TASK = "请检查 inbox，把 a.txt 移动到 archive，然后告诉我整理后的目录变化。"

try:
    # 如果是作为 .py 脚本运行
    current_dir = Path(__file__).resolve().parent
except NameError:
    # 如果是在 Jupyter Notebook 或交互式命令行中运行
    current_dir = Path.cwd()

WORKSPACE =current_dir / "demo_workspace"
print(f"工作目录: {WORKSPACE}")


In [ ]:
def reset_workspace() -> None:
    if WORKSPACE.exists():
        shutil.rmtree(WORKSPACE)
    (WORKSPACE / "inbox").mkdir(parents=True)
    (WORKSPACE / "archive").mkdir()
    (WORKSPACE / "inbox" / "a.txt").write_text(
        "Hello from  AgentLoop demo.",
        encoding="utf-8",
    )


def show_workspace() -> str:
    items = sorted(WORKSPACE.rglob("*"))
    lines = []
    for item in items:
        rel = item.relative_to(WORKSPACE).as_posix()
        lines.append(f"- {rel}/" if item.is_dir() else f"- {rel}")
    return "\n".join(lines) or "(empty)"


def workspace_path(path: str) -> Path:
    path = path.strip().replace("\\", "/")
    return WORKSPACE if path == "." else WORKSPACE / path.strip("/")


In [ ]:
@tool
def list_files(path: str = ".") -> str:
    """List files in the demo workspace."""
    target = workspace_path(path)
    if target.is_file():
        return target.relative_to(WORKSPACE).as_posix()
    files = sorted(item for item in target.rglob("*") if item.is_file())
    return "\n".join(f"- {item.relative_to(WORKSPACE).as_posix()}" for item in files) or "(empty)"


@tool
def move_file(source: str, target: str) -> str:
    """Move a file in the demo workspace."""
    source_path = workspace_path(source)
    target_path = workspace_path(target)
    if "." not in target_path.name:
        target_path = target_path / source_path.name
    target_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(source_path), str(target_path))
    return f"moved {source} -> {target_path.relative_to(WORKSPACE).as_posix()}"


In [ ]:
def load_llm() -> ChatOpenAI:
    load_dotenv(override=True)
    
    return ChatOpenAI(
        model = os.getenv("modelscope_model_id"),
        api_key = os.getenv("modelscope_key"),
        base_url = os.getenv("modelscope_url"),
        temperature = 0.0,
    )


In [ ]:
SYSTEM_PROMPT = """
你是一个文件整理助手。
你可以反复调用工具，直到完成用户任务。
推荐顺序：先 list_files("inbox")，再 move_file("inbox/a.txt", "archive/a.txt")，再 list_files(".")，最后总结。
每一轮最多调用一个工具。
"""


In [ ]:
def main() -> None:
    task = DEFAULT_TASK
    reset_workspace()

    print("=== 01. 手写 while AgentLoop ===")
    print("\n用户任务:")
    print(task)
    print("\n运行前 workspace:")
    print(show_workspace())

    tools = [list_files, move_file]
    tool_map = {item.name: item for item in tools}
    llm = load_llm().bind_tools(tools)
    
    messages = [
        SystemMessage(content=SYSTEM_PROMPT), 
        HumanMessage(content=task)
    ]

    for turn in range(1, 8):
        print(f"\n--- 第 {turn} 轮：模型思考 ---")
        response = llm.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            print("\n最终回答:")
            print(response.content)
            break

        tool_call = response.tool_calls[0]
        print("\n模型决定调用工具:")
        print(f"tool_name = {tool_call['name']}")
        print(f"tool_args = {tool_call['args']}")

        result = tool_map[tool_call["name"]].invoke(tool_call["args"])
        print("\n工具返回:")
        print(result)

        messages.append(
            ToolMessage(
                content=str(result),
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )

    print("\n运行后 workspace:")
    print(show_workspace())


In [ ]:
main()


### 2.1.2 LangChain ReAct实现

In [ ]:
def main() -> None:
    from langchain.agents import create_agent
    
    task = DEFAULT_TASK
    reset_workspace()

    print("=== 02. LangChain create_agent: 封装好的 ReActLoop ===")
    print("\n用户任务:")
    print(task)
    print("\n运行前 workspace:")
    print(show_workspace())
    
    agent = create_agent(
        model = load_llm(),
        tools = [list_files, move_file],
        system_prompt = SYSTEM_PROMPT,
    )
    
    result = agent.invoke({"messages": [{"role": "user", "content": task}]})
    
    print("\nAgent消息流：")
    for message in result["messages"]:
        print(f"\n[{getattr(message, "type", "unknown")}]")
        if getattr(message, "tool_calls", None):
            print(message.tool_calls)
        elif getattr(message, "content", ""):
            print(message.content)
    
    print("\n运行后 workspace:")
    print(show_workspace())


In [ ]:
main()


### 2.1.3 LangGraph ReAct 实现

In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path
from typing import TypedDict

from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph


In [ ]:
class AgentState(TypedDict):
    messages: list[BaseMessage]


In [ ]:
def build_agent_graph() -> StateGraph:
    tools = [list_files, move_file]
    tool_map = {item.name: item for item in tools}
    llm = load_llm().bind_tools(tools)
    
    def agent_node(state: AgentState) -> AgentState:
        print("\n[agent] 模型思考")
        response = llm.invoke(
            [SystemMessage(content=SYSTEM_PROMPT), *state["messages"]]
        )
        print(f"模型回复: {response.content}")
        return {"messages": [*state["messages"], response]}
    
    def tools_node(state: AgentState) -> AgentState:
        response = state["messages"][-1]
        new_messages = list(state["messages"])

        for tool_call in response.tool_calls:
            print("\n[tools] 执行工具")
            print(f"tool_name = {tool_call['name']}")
            print(f"tool_args = {tool_call['args']}")
            result = tool_map[tool_call["name"]].invoke(tool_call["args"])
            print(result)
            new_messages.append(
                ToolMessage(
                    content=str(result),
                    name=tool_call["name"],
                    tool_call_id=tool_call["id"],
                )
            )

        return {"messages": new_messages}
    
    def should_continue(state: AgentState) -> str:
        last_message = state["messages"][-1]
        return "tools" if getattr(last_message, "tool_calls", None) else END
    
    graph = StateGraph(AgentState)
    
    graph.add_node("agent", agent_node)
    graph.add_node("tools", tools_node)
    
    graph.add_edge(START, "agent")
    graph.add_conditional_edges("agent", should_continue)
    graph.add_edge("tools", "agent")

    return graph


In [ ]:
def main() -> None:
    task = DEFAULT_TASK
    reset_workspace()

    print("=== 03. LangGraph ReAct：agent node + tools node ===")
    print("\n用户任务:")
    print(task)
    print("\n运行前 workspace:")
    print(show_workspace())
    
    graph = build_agent_graph()

    result = graph.compile().invoke(
        {"messages": [HumanMessage(content=task)]},
        config={"recursion_limit": 12},
    )

    print("\n最终回答:")
    print(result["messages"][-1].content)
    print("\n运行后 workspace:")
    print(show_workspace())


In [ ]:
main()


## 2.2 Reflection Loop
解决“模型只完成，不看完成的对不对”的问题

+ `思考`：输入模型，输出调用
+ `行动`：执行Tool逻辑
+ `批判`：引入另一个Role批判
+ `反馈`：结果放回模型重复

### 2.2.1 LangGraph Reflection

In [ ]:
SYSTEM_PROMPT = """
你是一个 Reflection 文件整理助手。
目标：检查 inbox，移动 inbox/a.txt 到 archive/a.txt，查看整理后的目录，然后总结。
每一轮最多调用一个工具。
如果有 reflection note，请优先参考它决定下一步。
"""


In [ ]:
REFLECTION_PROMPT = """
你是 reviewer。根据最近的工具结果，给 agent 一句下一步建议。
目标：检查 inbox -> 移动 inbox/a.txt 到 archive/a.txt -> 查看整理后的目录 -> 总结。
如果已经看到 archive/a.txt，请提醒 agent 停止调用工具并总结。
只输出一句 reflection note。
"""


In [ ]:
class AgentState(TypedDict):
    messages: list[BaseMessage]
    reflection: str


In [ ]:
def build_agent_graph() -> StateGraph:
    
    tools = [list_files, move_file]
    tool_map = {item.name: item for item in tools}
    base_llm = load_llm()
    tool_llm = base_llm.bind_tools(tools)
    
    def agent_node(state: AgentState) -> AgentState:
        print("\n[agent] 模型思考")
        prompt = SYSTEM_PROMPT
        if state["reflection"]:
            prompt += f"\n\nReflection note: {state['reflection']}"
        response = tool_llm.invoke([SystemMessage(content=prompt), *state["messages"]])
        print(f"模型回复:\n {response.content}")
        return {"messages": [*state["messages"], response], "reflection": state["reflection"]}

    def tools_node(state: AgentState) -> AgentState:
        response = state["messages"][-1]
        new_messages = list(state["messages"])

        for tool_call in response.tool_calls:
            print("\n[tools] 执行工具")
            print(f"tool_name = {tool_call['name']}")
            print(f"tool_args = {tool_call['args']}")
            result = tool_map[tool_call["name"]].invoke(tool_call["args"])
            print(f"tool_results = {result}")
            new_messages.append(
                ToolMessage(
                    content=str(result),
                    name=tool_call["name"],
                    tool_call_id=tool_call["id"],
                )
            )

        return {"messages": new_messages, "reflection": state["reflection"]}

    def reflection_node(state: AgentState) -> AgentState:
        print("\n[reflection] 复盘工具结果")
        transcript = "\n".join(str(message.content) for message in state["messages"][-4:])
        note = base_llm.invoke(
            [
                SystemMessage(content=REFLECTION_PROMPT),
                HumanMessage(content=transcript),
            ]
        ).content
        print(note)
        return {"messages": state["messages"], "reflection": str(note)}

    def should_continue(state: AgentState) -> str:
        last_message = state["messages"][-1]
        return "tools" if getattr(last_message, "tool_calls", None) else END
    
    graph = StateGraph(AgentState)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", tools_node)
    graph.add_node("reflection", reflection_node)
    
    graph.add_edge(START, "agent")
    graph.add_conditional_edges("agent", should_continue)
    graph.add_edge("tools", "reflection")
    graph.add_edge("reflection", "agent")
    
    return graph


In [ ]:
def main() -> None:
    task = DEFAULT_TASK
    reset_workspace()

    print("=== 04. LangGraph Reflaction：agent node + tools node + reflection node ===")
    print("\n用户任务:")
    print(task)
    print("\n运行前 workspace:")
    print(show_workspace())
    
    graph = build_agent_graph()

    result = graph.compile().invoke(
        {"messages": [HumanMessage(content=task)], "reflection": ""},
        config={"recursion_limit": 12},
    )

    print("\n最终回答:")
    print(result["messages"][-1].content)
    print("\n运行后 workspace:")
    print(show_workspace())


In [ ]:
main()


## 2.3 Plan & Execute
模型过于短视，不能完成长程任务

+ 计划：分析制定TODO
+ 思考：输入模型，输出调用
+ 行动：执行Tool逻辑
+ 评价：查看Todo执行情况
+ 反馈：结果放回模型重复

In [ ]:
import re


In [ ]:
PLANNER_PROMPT = """
把用户任务拆成 3 个步骤。
必须覆盖：检查 inbox、移动 a.txt 到 archive、查看整理后的目录。
每行一个步骤，不要写额外解释。
"""


In [ ]:
SYSTEM_PROMPT = """
你是一个 ReAct executor。
你会收到计划，请按计划用工具一步步完成任务。
每一轮最多调用一个工具。
如果有 reflection note，请优先参考它决定下一步。
"""


In [ ]:
REFLECTION_PROMPT = """
你是 reviewer。根据计划和最近工具结果，给 executor 一句下一步建议。
如果已经看到 archive/a.txt，请提醒 executor 停止调用工具并总结。
只输出一句 reflection note。
"""


In [ ]:
class AgentState(TypedDict):
    task: str
    plan: list[str]
    messages: list[BaseMessage]
    reflection: str


In [ ]:
def parse_plan(text: str) -> list[str]:
    steps = []
    for line in text.splitlines():
        line = re.sub(r"^\s*[-*\d.、)]+\s*", "", line).strip()
        if line:
            steps.append(line)
    return steps or ["检查 inbox", "移动 a.txt 到 archive", "查看整理后的目录"]


In [ ]:
def build_agent_graph() -> StateGraph:
    
    tools = [list_files, move_file]
    tool_map = {item.name: item for item in tools}
    base_llm = load_llm()
    tool_llm = base_llm.bind_tools(tools)
    
    def planner_node(state: AgentState) -> AgentState:
        print("\n[planner] 生成计划")
        response = base_llm.invoke(
            [SystemMessage(content=PLANNER_PROMPT), HumanMessage(content=state["task"])]
        )

        plan = parse_plan(str(response.content))

        for index, step in enumerate(plan, start=1):
            print(f"{index}. {step}")

        plan_text = "\n".join(f"{index}. {step}" for index, step in enumerate(plan, start=1))

        return {
            **state,
            "plan": plan,
            "messages": [HumanMessage(content=f"用户任务：{state['task']}\n\n计划：\n{plan_text}")],
        }

    def agent_node(state: AgentState) -> AgentState:
        print("\n[agent] 按计划执行下一步")
        prompt = SYSTEM_PROMPT

        if state["reflection"]:
            prompt += f"\n\nReflection note: {state['reflection']}"

        response = tool_llm.invoke([SystemMessage(content=prompt), *state["messages"]])
        return {**state, "messages": [*state["messages"], response]}

    def tools_node(state: AgentState) -> AgentState:
        response = state["messages"][-1]
        new_messages = list(state["messages"])

        for tool_call in response.tool_calls:
            print("\n[tools] 执行工具")
            print(f"tool_name = {tool_call['name']}")
            print(f"tool_args = {tool_call['args']}")
            result = tool_map[tool_call["name"]].invoke(tool_call["args"])
            print(result)
            new_messages.append(
                ToolMessage(
                    content=str(result),
                    name=tool_call["name"],
                    tool_call_id=tool_call["id"],
                )
            )

        return {**state, "messages": new_messages}
    
    def reflection_node(state: AgentState) -> AgentState:
        print("\n[reflection] 复盘计划和工具结果")
        plan_text = "\n".join(state["plan"])
        transcript = "\n".join(str(message.content) for message in state["messages"][-4:])
        note = base_llm.invoke(
            [
                SystemMessage(content=REFLECTION_PROMPT),
                HumanMessage(content=f"计划：\n{plan_text}\n\n最近记录：\n{transcript}"),
            ]
        ).content
        print(note)
        return {**state, "reflection": str(note)}

    def should_continue(state: AgentState) -> str:
        last_message = state["messages"][-1]
        return "tools" if getattr(last_message, "tool_calls", None) else END

    graph = StateGraph(AgentState)
    graph.add_node("planner", planner_node)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", tools_node)
    graph.add_node("reflection", reflection_node)
    
    graph.add_edge(START, "planner")
    graph.add_edge("planner", "agent")
    graph.add_conditional_edges("agent", should_continue)
    graph.add_edge("tools", "reflection")
    graph.add_edge("reflection", "agent")
    
    return graph


In [ ]:
def main() -> None:
    task = DEFAULT_TASK
    reset_workspace()

    print("=== 05. LangGraph Plan-Execute：plan node + agent node + tools node + reflection node ===")
    print("\n用户任务:")
    print(task)
    print("\n运行前 workspace:")
    print(show_workspace())
    
    graph = build_agent_graph()

    result = graph.compile().invoke(
        {"task": task, "plan": [], "messages": [], "reflection": ""},
        config={"recursion_limit": 50},
    )

    print("\n最终回答:")
    print(result["messages"][-1].content)
    print("\n运行后 workspace:")
    print(show_workspace())


In [ ]:
main()


## 2.4 Multi-agent

In [ ]:
class MultiAgentState(TypedDict):
    task: str
    next_agent: str
    file_report: str
    code_report: str
    final_answer: str


In [ ]:
SUMMARY_PROMPT = """
你是 supervisor agent。
根据 file_agent 和 code_agent 的报告，用中文做一个简短总结。
"""

In [ ]:
def build_agent_graph() -> StateGraph:
    llm = load_llm()
    
    file_agent = create_agent(
        llm,
        tools=[list_files, move_file],
        system_prompt=FILE_AGENT_PROMPT,
        name="file_agent",
    )
    code_agent = create_agent(
        llm,
        tools=[write_file],
        system_prompt=CODE_AGENT_PROMPT,
        name="code_agent",
    )
    
    def supervisor_node(state: MultiAgentState) -> MultiAgentState:
        print("\n[supervisor] 决定下一个 agent")

        if not state["file_report"]:
            next_agent = "file_agent"
        elif not state["code_report"]:
            next_agent = "code_agent"
        else:
            next_agent = "finish"

        decision = llm.invoke(
            [
                SystemMessage(content=SUPERVISOR_PROMPT),
                HumanMessage(
                    content=(
                        f"用户任务：{state['task']}\n\n"
                        f"file_report：{state['file_report'] or '(empty)'}\n\n"
                        f"code_report：{state['code_report'] or '(empty)'}\n\n"
                        f"请判断下一个 agent。建议答案：{next_agent}"
                    )
                ),
            ]
        ).content

        decision = str(decision).strip()
        if decision not in {"file_agent", "code_agent", "finish"}:
            decision = next_agent
        print(decision)

        if decision == "finish":
            final_answer = llm.invoke(
                [
                    SystemMessage(content=SUMMARY_PROMPT),
                    HumanMessage(
                        content=(
                            f"file_agent:\n{state['file_report']}\n\n"
                            f"code_agent:\n{state['code_report']}"
                        )
                    ),
                ]
            ).content
            return {**state, "next_agent": "finish", "final_answer": str(final_answer)}

        return {**state, "next_agent": decision}
    
    def file_agent_node(state: MultiAgentState) -> MultiAgentState:
        print("\n[file_agent node] 调用真正的 file_agent")
        result = file_agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": (
                            f"用户原始任务：{state['task']}\n\n"
                            "你是 file_agent，只处理其中和文件整理有关的部分。"
                        ),
                    }
                ]
            }
        )
        report = last_message_text(result)
        print(report)
        return {**state, "file_report": report}
    
    def code_agent_node(state: MultiAgentState) -> MultiAgentState:
        print("\n[code_agent node] 调用真正的 code_agent")
        result = code_agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": (
                            f"用户原始任务：{state['task']}\n\n"
                            f"file_agent 已完成的结果：\n{state['file_report']}\n\n"
                            "你是 code_agent，只处理其中和代码生成有关的部分。"
                        ),
                    }
                ]
            }
        )
        report = last_message_text(result)
        print(report)
        return {**state, "code_report": report}
    
    def route_next(state: MultiAgentState) -> str:
        if state["next_agent"] == "file_agent":
            return "file_agent"
        if state["next_agent"] == "code_agent":
            return "code_agent"
        return END
    
    graph = StateGraph(MultiAgentState)
    graph.add_node("supervisor", supervisor_node)
    graph.add_node("file_agent", file_agent_node)
    graph.add_node("code_agent", code_agent_node)

    graph.add_edge(START, "supervisor")
    graph.add_conditional_edges("supervisor", route_next)
    graph.add_edge("file_agent", "supervisor")
    graph.add_edge("code_agent", "supervisor")
    
    return graph
    

In [ ]:
def main() -> None:
    task = DEFAULT_TASK
    reset_workspace()

    print("=== 02. MultiAgent：每个 Agent 是一个 Node，用 conditional_edge 路由 ===")
    print("\n用户任务:")
    print(task)
    print("\n运行前 workspace:")
    print(show_workspace())
    
    graph = build_agent_graph()

    result = graph.compile().invoke(
        {
            "task": task,
            "next_agent": "",
            "file_report": "",
            "code_report": "",
            "final_answer": "",
        },
        config={"recursion_limit": 12},
    )

    print("\n[supervisor] 最终回答:")
    print(result["final_answer"])
    print("\n运行后 workspace:")
    print(show_workspace())


In [ ]:
main()